[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-4-4-building-rest-apis-with-fastapi.ipynb)

# Lesson 4.4 · Building REST APIs with FastAPI

**Free Python For Development** · Topic 4: File I/O, Databases & REST APIs

_FastAPI, Pydantic, middleware, OpenAPI._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-4-4)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

**Setup** — install the libraries this lesson uses (takes a few seconds):

In [ ]:
!pip install -q fastapi httpx httpx2

## Introduction & Core Concepts of Building REST APIs with FastAPI

Imagine you want to build a modern web application that lets users interact with your data in real time — a todo list, a weather dashboard, or even a social platform. At the heart of these apps sits a **REST API**: a messenger that carries requests from the user’s browser (or mobile app) to your backend code and returns responses.

**FastAPI** is one of Python’s most popular frameworks for this job. It is extremely fast, easy to write, and uses Python’s type hints to validate data automatically and generate interactive documentation.

> 💡 **Key insight:** An endpoint is simply a Python function decorated with a path and HTTP method. FastAPI turns that function into a web resource that clients can call.

### A Minimal Todo API

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Todo API")

class Todo(BaseModel):
    title: str
    completed: bool = False

# In-memory store for the demo
todos: list[Todo] = []

@app.get("/todos")
def list_todos():
    return todos

@app.post("/todos", status_code=201)
def create_todo(todo: Todo):
    todos.append(todo)
    return todo

# On your computer, run with: uvicorn main:app --reload
# In Colab, call it with TestClient in the next cell.

**Try the API** — `TestClient` sends real HTTP requests to the app without starting a server:

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
print(client.post("/todos", json={"title": "Learn FastAPI"}).json())
print(client.get("/todos").json())

**What happens**

- `FastAPI()` creates the application instance.
- `@app.get` / `@app.post` register routes.
- The `Todo` model defines the expected shape of the data.
- FastAPI automatically parses JSON bodies into that model.
- Returning a model (or list) becomes a JSON response.

> 💡 **Mental model:** Think of each decorated function as a door. Clients knock on the door with an HTTP request; FastAPI opens it, checks the data, runs your function, and hands back a response.

## Pydantic Validation & Middleware

FastAPI relies on **Pydantic** for data validation. Pydantic acts like a security guard at the front door: every incoming payload is checked against the declared types. If a required field is missing or the wrong type is sent, the request is rejected with a clear error before your business logic ever runs.

**Pydantic Models**

Define the shape of request and response bodies. Type hints become runtime validation rules and also drive the auto-generated OpenAPI schema.

**Middleware**

Code that runs on every request (or response). Perfect for logging, authentication, CORS, timing, or adding custom headers.

### Validation + Simple Logging Middleware

In [ ]:
from fastapi import FastAPI, Request
from pydantic import BaseModel, Field
import time

app = FastAPI()

class TodoCreate(BaseModel):
    title: str = Field(..., min_length=1, max_length=100)
    completed: bool = False

@app.middleware("http")
async def log_requests(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    duration = (time.perf_counter() - start) * 1000
    print(f"{request.method} {request.url.path} → {response.status_code} ({duration:.1f} ms)")
    return response

@app.post("/todos")
def create(todo: TodoCreate):
    # todo is already validated
    return {"id": 1, **todo.model_dump()}

# Invalid payload example:
# {"title": ""}  → 422 Unprocessable Entity

**Try the API** — `TestClient` sends real HTTP requests to the app without starting a server:

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
print(client.post("/todos", json={"title": "Write tests"}).json())

bad = client.post("/todos", json={"title": ""})   # fails validation
print(bad.status_code, bad.json()["detail"][0]["msg"])

**Key points**

- `Field(...)` marks a value as required and adds constraints.
- Invalid data never reaches the path operation function.
- Middleware wraps the whole request/response cycle.
- You can stack multiple middleware layers.

> 💡 **Architecture tip:** Keep path-operation functions thin. Put validation in Pydantic models, cross-cutting concerns in middleware, and business logic in separate service modules.

## Automatic Documentation & Best Practices

One of FastAPI’s most loved features is automatic **OpenAPI** documentation. As soon as the server starts you get two interactive pages:

- `/docs` — Swagger UI (try endpoints right in the browser)
- `/redoc` — ReDoc (clean, readable reference)

These pages are generated from your type hints and Pydantic models, so they stay perfectly in sync with the code.

### Complete Mini-API with Docs Ready

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(
    title="Todo API",
    description="A tiny but complete example",
    version="1.0.0"
)

class Todo(BaseModel):
    id: int
    title: str
    completed: bool = False

class TodoCreate(BaseModel):
    title: str
    completed: bool = False

db: dict[int, Todo] = {}
next_id = 1

@app.post("/todos", response_model=Todo, status_code=201)
def create_todo(payload: TodoCreate):
    global next_id
    todo = Todo(id=next_id, **payload.model_dump())
    db[next_id] = todo
    next_id += 1
    return todo

@app.get("/todos/{todo_id}", response_model=Todo)
def get_todo(todo_id: int):
    if todo_id not in db:
        raise HTTPException(status_code=404, detail="Todo not found")
    return db[todo_id]

@app.get("/todos", response_model=list[Todo])
def list_todos():
    return list(db.values())

**Try the API** — `TestClient` sends real HTTP requests to the app without starting a server:

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
print(client.post("/todos", json={"title": "Ship the API"}).json())
print(client.get("/todos/1").json())
missing = client.get("/todos/99")
print(missing.status_code, missing.json())
print(client.get("/todos").json())

**Production habits**

- Always declare `response_model` for clear contracts.
- Use `HTTPException` for expected error cases.
- Keep the OpenAPI title/description up to date.
- Prefer dependency injection for shared resources (DB sessions, auth).
- Run behind a production ASGI server (Uvicorn + Gunicorn or Hypercorn).

> 💡 **Big takeaway:** FastAPI gives you speed, automatic data validation with Pydantic, powerful middleware, and interactive OpenAPI documentation out of the box. These features let you build robust, scalable REST APIs in Python with far less boilerplate than older frameworks.

---

✅ **Lesson complete!** Next up: **Lesson 5.1 · Exceptions & Custom Errors** — [watch it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-5-1) or open `lesson-5-1-exceptions-and-custom-errors.ipynb`.

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).